# 07 — Merging & Joining
**Goal:** Combine Adobe Analytics data with paid platform data into a single unified DataFrame.

This is the core of cross-source analysis — the thing Adobe's own dashboard can't do.

Topics:
- `pd.merge()` — SQL-style joins
- Join types: inner, left, right, outer
- `pd.concat()` — stack DataFrames vertically or horizontally
- Handling key mismatches
- Real-world: Adobe + paid unified table

In [1]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import pandas as pd
import numpy as np
from pathlib import Path

# Simulate Adobe Analytics data (daily, all channels)
np.random.seed(42)
dates = pd.date_range('2024-01-01', '2024-01-10')

adobe = pd.DataFrame({
    'date':         dates,
    'visits':       np.random.randint(2500, 4000, len(dates)),
    'activations':  np.random.randint(50, 120, len(dates)),
    'bounce_rate':  np.random.uniform(0.35, 0.55, len(dates)).round(3),
    'avg_time_sec': np.random.randint(120, 300, len(dates)),
})

# Simulate Paid data (daily, only paid channel)
paid = pd.DataFrame({
    'date':         dates[:8],   # only 8 days — simulates missing data
    'spend':        np.random.uniform(800, 1500, 8).round(2),
    'impressions':  np.random.randint(50000, 150000, 8),
    'clicks':       np.random.randint(1000, 3000, 8),
    'paid_conversions': np.random.randint(15, 50, 8),
})

print('Adobe shape:', adobe.shape)
print('Paid shape: ', paid.shape)
print()
print(adobe)
print()
print(paid)

Adobe shape: (10, 5)
Paid shape:  (8, 5)

        date  visits  activations  bounce_rate  avg_time_sec
0 2024-01-01    3626           73        0.387           294
1 2024-01-02    3959           52        0.411           170
2 2024-01-03    3360           71        0.455           227
3 2024-01-04    3794          102        0.436           174
4 2024-01-05    3630           51        0.408           183
5 2024-01-06    3595           79        0.472           250
6 2024-01-07    3544           87        0.378           170
7 2024-01-08    2621           51        0.408           254
8 2024-01-09    2966          113        0.423           140
9 2024-01-10    3738          109        0.441           192

        date    spend  impressions  clicks  paid_conversions
0 2024-01-01   845.54        98555    2025                32
1 2024-01-02  1464.22        67159    2021                40
2 2024-01-03  1475.94       130077    2413                48
3 2024-01-04  1365.88        85920    1565

## 1. Join types — understanding the difference

```
inner  → only rows that exist in BOTH tables
left   → all rows from left table, NaN where right has no match
right  → all rows from right table, NaN where left has no match  
outer  → all rows from BOTH tables, NaN where either side has no match
```

In [2]:
# inner join — only days that exist in both adobe AND paid (8 days)
inner = pd.merge(adobe, paid, on='date', how='inner')
print('Inner join:', len(inner), 'rows')
print(inner[['date','visits','spend','paid_conversions']])

Inner join: 8 rows
        date  visits    spend  paid_conversions
0 2024-01-01    3626   845.54                32
1 2024-01-02    3959  1464.22                40
2 2024-01-03    3360  1475.94                48
3 2024-01-04    3794  1365.88                24
4 2024-01-05    3630  1013.23                28
5 2024-01-06    3595   868.37                45
6 2024-01-07    3544  1278.96                29
7 2024-01-08    2621  1108.11                22


In [3]:
# left join — keep all adobe rows, fill missing paid data with NaN
# Use this most of the time — you don't want to lose Adobe data
left = pd.merge(adobe, paid, on='date', how='left')
print('Left join:', len(left), 'rows')
print(left[['date','visits','spend','paid_conversions']])
print()
print('Nulls in spend:', left['spend'].isnull().sum())  # 2 days have no paid data

Left join: 10 rows
        date  visits    spend  paid_conversions
0 2024-01-01    3626   845.54              32.0
1 2024-01-02    3959  1464.22              40.0
2 2024-01-03    3360  1475.94              48.0
3 2024-01-04    3794  1365.88              24.0
4 2024-01-05    3630  1013.23              28.0
5 2024-01-06    3595   868.37              45.0
6 2024-01-07    3544  1278.96              29.0
7 2024-01-08    2621  1108.11              22.0
8 2024-01-09    2966      NaN               NaN
9 2024-01-10    3738      NaN               NaN

Nulls in spend: 2


In [4]:
# outer join — all rows from both, NaN where no match
outer = pd.merge(adobe, paid, on='date', how='outer')
print('Outer join:', len(outer), 'rows')
print(outer[['date','visits','spend']])

Outer join: 10 rows
        date  visits    spend
0 2024-01-01    3626   845.54
1 2024-01-02    3959  1464.22
2 2024-01-03    3360  1475.94
3 2024-01-04    3794  1365.88
4 2024-01-05    3630  1013.23
5 2024-01-06    3595   868.37
6 2024-01-07    3544  1278.96
7 2024-01-08    2621  1108.11
8 2024-01-09    2966      NaN
9 2024-01-10    3738      NaN


## 2. Merge on multiple keys
When joining by date AND channel (common in channel-level analysis)

In [5]:
# Simulate channel-level Adobe data
adobe_ch = pd.DataFrame({
    'date':    pd.to_datetime(['2024-01-01','2024-01-01','2024-01-02','2024-01-02']),
    'channel': ['organic','paid','organic','paid'],
    'visits':  [1200, 900, 1100, 850],
    'activations': [36, 18, 33, 17],
})

# Simulate paid platform data (only for paid channel)
paid_ch = pd.DataFrame({
    'date':    pd.to_datetime(['2024-01-01','2024-01-02']),
    'channel': ['paid','paid'],
    'spend':   [450.0, 420.0],
    'clicks':  [1800, 1700],
})

# Merge on both date and channel
merged = pd.merge(adobe_ch, paid_ch, on=['date','channel'], how='left')
print(merged)

        date  channel  visits  activations  spend  clicks
0 2024-01-01  organic    1200           36    NaN     NaN
1 2024-01-01     paid     900           18  450.0  1800.0
2 2024-01-02  organic    1100           33    NaN     NaN
3 2024-01-02     paid     850           17  420.0  1700.0


## 3. Handling key mismatches — different column names

In [6]:
# Sometimes the join key has a different name in each table
# Adobe uses 'date', paid uses 'day'
paid_renamed = paid.rename(columns={'date': 'day'})

merged = pd.merge(
    adobe, paid_renamed,
    left_on='date',    # key in left table
    right_on='day',    # key in right table
    how='left'
)

# Drop the duplicate key column
merged = merged.drop(columns=['day'])
print(merged[['date','visits','spend']].head())

        date  visits    spend
0 2024-01-01    3626   845.54
1 2024-01-02    3959  1464.22
2 2024-01-03    3360  1475.94
3 2024-01-04    3794  1365.88
4 2024-01-05    3630  1013.23


## 4. `pd.concat()` — stack DataFrames

In [7]:
# Stack vertically — combine data from multiple months/files
jan = pd.DataFrame({'date': pd.date_range('2024-01-01', periods=3), 'visits': [3200,2800,3500]})
feb = pd.DataFrame({'date': pd.date_range('2024-02-01', periods=3), 'visits': [3100,2900,3400]})
mar = pd.DataFrame({'date': pd.date_range('2024-03-01', periods=3), 'visits': [3300,3000,3600]})

# ignore_index=True resets the index to 0,1,2,...
full = pd.concat([jan, feb, mar], ignore_index=True)
print('Vertical concat:')
print(full)

Vertical concat:
        date  visits
0 2024-01-01    3200
1 2024-01-02    2800
2 2024-01-03    3500
3 2024-02-01    3100
4 2024-02-02    2900
5 2024-02-03    3400
6 2024-03-01    3300
7 2024-03-02    3000
8 2024-03-03    3600


In [8]:
# Stack horizontally — add columns from another DataFrame (must have same index)
df_a = pd.DataFrame({'visits': [3200, 2800]}, index=[0,1])
df_b = pd.DataFrame({'spend':  [450.0, 420.0]}, index=[0,1])

combined = pd.concat([df_a, df_b], axis=1)   # axis=1 = side by side
print('Horizontal concat:')
print(combined)

Horizontal concat:
   visits  spend
0    3200  450.0
1    2800  420.0


## 5. Real-world: Full Adobe + paid unified pipeline

In [9]:
# Load funnel data as our "Adobe" source
df_funnel = pd.read_csv('data/raw/funnel_data.csv', parse_dates=['date'])

# Aggregate to daily total (all channels)
adobe_daily = df_funnel.groupby('date').agg(
    visits      = ('visita_landing',     'sum'),
    activations = ('activacion_tarjeta',  'sum'),
    otp_total   = ('otp',                'sum'),
).reset_index()

# Simulate paid daily data
np.random.seed(42)
dates = adobe_daily['date']
paid_daily = pd.DataFrame({
    'date':       dates,
    'spend':      np.random.uniform(800, 1800, len(dates)).round(2),
    'impressions':np.random.randint(80000, 200000, len(dates)),
    'clicks':     np.random.randint(1500, 4000, len(dates)),
    'paid_conv':  np.random.randint(10, 45, len(dates)),
})

# Merge
unified = pd.merge(adobe_daily, paid_daily, on='date', how='left')

# Derived metrics combining both sources
unified = unified.assign(
    cvr          = lambda x: x['activations'] / x['visits'] * 100,
    cpa          = lambda x: x['spend'] / x['paid_conv'],
    ctr          = lambda x: x['clicks'] / x['impressions'] * 100,
    roas         = lambda x: (x['paid_conv'] * 500) / x['spend'],  # assume $500 LTV
    paid_share   = lambda x: x['paid_conv'] / x['activations'] * 100,
)

print('Unified table shape:', unified.shape)
print(unified[['date','visits','activations','cvr','spend','cpa','roas']].head(10).round(2))

Unified table shape: (90, 13)
        date  visits  activations   cvr    spend     cpa   roas
0 2024-01-01    3097           69  2.23  1174.54   27.97  17.88
1 2024-01-02    2851           64  2.24  1750.71  134.67   3.71
2 2024-01-03    2978           65  2.18  1531.99   36.48  13.71
3 2024-01-04    2845           61  2.14  1398.66   60.81   8.22
4 2024-01-05    2914           64  2.20   956.02   31.87  15.69
5 2024-01-06    2565           56  2.18   955.99   32.97  15.17
6 2024-01-07    2554           54  2.11   858.08   50.48   9.91
7 2024-01-08    3049           68  2.23  1666.18  104.14   4.80
8 2024-01-09    2892           62  2.14  1401.12  116.76   4.28
9 2024-01-10    3043           68  2.23  1508.07   58.00   8.62


/var/folders/3m/z0qp5nwj1vlccby390hqt0kc0000gn/T/ipykernel_10077/241002389.py:35: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(unified[['date','visits','activations','cvr','spend','cpa','roas']].head(10).round(2))


In [10]:
# Save the unified table — this is the clean input for all your charts
from pathlib import Path
Path('data/clean').mkdir(exist_ok=True)
unified.to_parquet('data/clean/unified_daily.parquet', index=False)
unified.to_csv('data/clean/unified_daily.csv', index=False)
print('Saved unified_daily to data/clean/')

Saved unified_daily to data/clean/


## Summary
| Tool | Use when |
|---|---|
| `merge(left, right, on='date', how='left')` | Join two tables on a key |
| `how='inner'` | Keep only matching rows |
| `how='left'` | Keep all left rows (most common) |
| `how='outer'` | Keep all rows from both |
| `left_on=` / `right_on=` | Join keys with different names |
| `on=['date','channel']` | Join on multiple keys |
| `pd.concat([df1,df2])` | Stack rows from multiple DataFrames |
| `pd.concat([df1,df2], axis=1)` | Add columns side by side |

**Next:** `08_time_series.ipynb` — resample, rolling, shift.